# Multi-Model Ensemble with CLAHE + AutoAugment

Pipeline for fashion classification using diverse pretrained models with ensemble approach:
- **Preprocessing**: CLAHE (Contrast Limited Adaptive Histogram Equalization) for adaptive contrast enhancement
- **Augmentation**: AutoAugment data augmentation (training only)
- **Architectures**: ResNet-50, ResNet-18, EfficientNet-B0, MobileNet-V2, DenseNet-121, ShuffleNet-V2
- **Strategy**: Separate models for jenis and warna (12 models total)
- **Ensemble**: Majority voting ensemble
- **Evaluation**: Exact Match Ratio (EMR)

## 1. Setup and Dependencies

In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from torchvision.io import read_image
from torchvision.transforms import AutoAugment
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
print('Models to be used in ensemble:')
print('1. ResNet-50 - Deep residual network (50 layers)')
print('2. ResNet-18 - Lighter residual network (18 layers)')
print('3. EfficientNet-B0 - Compound scaling efficient network')
print('4. MobileNet-V2 - Mobile-optimized depthwise separable convolutions')
print('5. DenseNet-121 - Dense connections between layers')
print('6. ShuffleNet-V2 - Channel shuffle for efficient computation')
print('\nTotal: 12 models (6 for jenis, 6 for warna)')

## 2. Dataset with CLAHE + AutoAugment

In [ ]:
class CLAHETransform:
    """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)"""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
    
    def __call__(self, img):
        # Convert PIL to numpy array
        if isinstance(img, Image.Image):
            img_np = np.array(img)
        else:
            img_np = img
        
        # Convert RGB to LAB color space
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        
        # Split LAB channels
        l, a, b = cv2.split(lab)
        
        # Apply CLAHE to L channel
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        l_clahe = clahe.apply(l)
        
        # Merge channels
        lab_clahe = cv2.merge([l_clahe, a, b])
        
        # Convert back to RGB
        rgb_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
        
        # Convert back to PIL Image
        return Image.fromarray(rgb_clahe)

class FashionDataset(torch.utils.data.Dataset):
    def __init__(self, df_path, img_path, transform=None, is_train=True):
        self.df = pd.read_csv(df_path, index_col='id')
        self.img_path = img_path
        self.transform = transform
        self.auto_augment = AutoAugment()
        self.clahe_transform = CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8))
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.index[idx]
        
        for ext in ['.jpg', '.png']:
            img_file = os.path.join(self.img_path, f'{img_id}{ext}')
            if os.path.exists(img_file):
                break
        
        # Read image as PIL Image
        img = Image.open(img_file).convert('RGB')
        
        # Apply CLAHE for adaptive contrast enhancement
        img = self.clahe_transform(img)
        
        # Apply AutoAugment only during training
        if self.is_train:
            # Convert to tensor for AutoAugment
            img_tensor = transforms.ToTensor()(img)
            img_tensor = (img_tensor * 255).byte()
            img_tensor = self.auto_augment(img_tensor)
            # Convert back to PIL
            img = transforms.ToPILImage()(img_tensor)
        
        # Apply final transforms
        if self.transform:
            img = self.transform(img)
        
        jenis = self.df.iloc[idx]['jenis']
        warna = self.df.iloc[idx]['warna']
        
        return img, jenis, warna

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = FashionDataset('train.csv', 'train/train', transform, is_train=True)
val_dataset = FashionDataset('train.csv', 'train/train', transform, is_train=False)

train_idx, val_idx = train_test_split(
    list(range(len(train_dataset))), test_size=0.2, random_state=42,
    stratify=[train_dataset[i][1] for i in range(len(train_dataset))]
)

train_subset = torch.utils.data.Subset(train_dataset, train_idx)
val_subset = torch.utils.data.Subset(val_dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=0)

print(f'Train: {len(train_subset)}, Val: {len(val_subset)}')
print('Preprocessing: CLAHE (clip_limit=2.0, tile_grid_size=(8,8)) + AutoAugment (train only)')

## 2.1. Class Distribution Analysis

In [ ]:
# Analyze class distribution
df_train = pd.read_csv('train.csv')

print('='*60)
print('CLASS DISTRIBUTION ANALYSIS')
print('='*60)

# Jenis distribution
print('\nJENIS Distribution:')
jenis_counts = df_train['jenis'].value_counts().sort_index()
print(jenis_counts)
print(f'Jenis 0: {jenis_counts[0]} ({jenis_counts[0]/len(df_train)*100:.2f}%)')
print(f'Jenis 1: {jenis_counts[1]} ({jenis_counts[1]/len(df_train)*100:.2f}%)')
jenis_ratio = max(jenis_counts) / min(jenis_counts)
print(f'Imbalance Ratio: {jenis_ratio:.2f}:1')

# Warna distribution
print('\nWARNA Distribution:')
warna_counts = df_train['warna'].value_counts().sort_index()
print(warna_counts)
for i in range(5):
    print(f'Warna {i}: {warna_counts[i]} ({warna_counts[i]/len(df_train)*100:.2f}%)')
warna_ratio = max(warna_counts) / min(warna_counts)
print(f'Imbalance Ratio: {warna_ratio:.2f}:1')

# Combined distribution
print('\nCOMBINED (Jenis + Warna) Distribution:')
combined = df_train.groupby(['jenis', 'warna']).size()
print(combined)

print('='*60)

In [ ]:
# Calculate class weights for imbalanced data handling
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights for JENIS
jenis_labels = df_train['jenis'].values
jenis_class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(jenis_labels),
    y=jenis_labels
)
jenis_weights = torch.FloatTensor(jenis_class_weights).to(device)

print('\nJENIS Class Weights:')
for i, weight in enumerate(jenis_class_weights):
    print(f'  Class {i}: {weight:.4f}')

# Compute class weights for WARNA
warna_labels = df_train['warna'].values
warna_class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(warna_labels),
    y=warna_labels
)
warna_weights = torch.FloatTensor(warna_class_weights).to(device)

print('\nWARNA Class Weights:')
for i, weight in enumerate(warna_class_weights):
    print(f'  Class {i}: {weight:.4f}')

print('\nClass weights computed and ready for training!')

## 2.2. Balanced Sampling Strategy

In [ ]:
# Create WeightedRandomSampler for balanced mini-batches
# This ensures each batch has balanced representation of classes

# Get labels for all samples
jenis_labels_all = df_train['jenis'].values
warna_labels_all = df_train['warna'].values

# Calculate class counts for the entire dataset
jenis_class_counts = np.bincount(jenis_labels_all)
warna_class_counts = np.bincount(warna_labels_all)

print(f"Jenis class counts: {jenis_class_counts}")
print(f"Warna class counts: {warna_class_counts}")

# Calculate sample weights for each sample (inverse of class frequency)
jenis_sample_weights = 1.0 / jenis_class_counts[jenis_labels_all]
warna_sample_weights = 1.0 / warna_class_counts[warna_labels_all]

# Combine weights (multiply for joint distribution balancing)
combined_sample_weights = jenis_sample_weights * warna_sample_weights

# Normalize weights to sum to 1
combined_sample_weights = combined_sample_weights / combined_sample_weights.sum() * len(combined_sample_weights)

print(f"\nSample weights statistics (before filtering to train set):")
print(f"  Range: {combined_sample_weights.min():.4f} - {combined_sample_weights.max():.4f}")
print(f"  Mean: {combined_sample_weights.mean():.4f}")
print(f"  Total samples: {len(combined_sample_weights)}")

# Convert to tensor (we'll filter to train_idx in the next cell)
sample_weights_tensor = torch.FloatTensor(combined_sample_weights)

print(f"\nThis sampler will be used to create balanced mini-batches during training")
print(f"Minority class samples will be selected more frequently in each epoch")

In [ ]:
# Create balanced sampler for training
# Get sample weights for train indices only
train_sample_weights = sample_weights_tensor[train_idx]

print(f"Train sample weights statistics:")
print(f"  Range: {train_sample_weights.min():.4f} - {train_sample_weights.max():.4f}")
print(f"  Mean: {train_sample_weights.mean():.4f}")
print(f"  Std: {train_sample_weights.std():.4f}")

# Create WeightedRandomSampler
train_sampler = WeightedRandomSampler(
    weights=train_sample_weights,
    num_samples=len(train_sample_weights),
    replacement=True  # Allow same sample to be selected multiple times per epoch
)

# Recreate train_loader with sampler (note: shuffle must be False when using sampler)
train_loader_balanced = DataLoader(
    train_subset, 
    batch_size=64, 
    sampler=train_sampler,  # Using sampler instead of shuffle
    num_workers=0
)

print(f"\n✓ Created balanced train_loader with WeightedRandomSampler")
print(f"✓ Training samples: {len(train_sample_weights)}")
print(f"✓ Minority classes will be oversampled during training")
print(f"✓ Each epoch will have balanced representation of all classes")
print(f"\nDataLoader Options:")
print(f"  - train_loader: shuffle=True (imbalanced, standard)")
print(f"  - train_loader_balanced: WeightedRandomSampler (balanced, recommended)")

## 3. Model Architectures

In [ ]:
class ResNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        resnet = models.resnet50(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.fc = nn.Linear(resnet.fc.in_features, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class EfficientNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        efficientnet = models.efficientnet_b0(pretrained=True)
        self.features = efficientnet.features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(efficientnet.classifier[1].in_features, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class MobileNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = mobilenet.features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(mobilenet.classifier[1].in_features, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class ResNet18Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        resnet18 = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet18.children())[:-1])
        self.fc = nn.Linear(resnet18.fc.in_features, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class DenseNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        densenet = models.densenet121(pretrained=True)
        self.features = densenet.features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(densenet.classifier.in_features, num_classes)
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

class ShuffleNetModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        shufflenet = models.shufflenet_v2_x1_0(pretrained=True)
        # Extract only the conv and maxpool layers, not the fc
        self.conv1 = shufflenet.conv1
        self.maxpool = shufflenet.maxpool
        self.stage2 = shufflenet.stage2
        self.stage3 = shufflenet.stage3
        self.stage4 = shufflenet.stage4
        self.conv5 = shufflenet.conv5
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        # Get the correct number of output channels from conv5
        in_features = shufflenet.fc.in_features
        self.fc = nn.Linear(in_features, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.conv5(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

print('Model classes defined')

## 4. Training Function with Model Checkpointing

### Training Configuration:
- **Optimizer**: AdamW (lr=0.0001, weight_decay=1e-5)
- **LR Scheduler**: ReduceLROnPlateau (factor=0.5, patience=5, min_lr=1e-6)
- **Early Stopping**: Patience=10 epochs without improvement
- **Model Checkpointing**: Best model saved based on validation accuracy
- **Checkpoint Location**: `saved_models/{model_name}_{task}_best.pth`
- **Class Weighting**: Weighted CrossEntropyLoss for imbalanced data handling

In [ ]:
def train_model(model, train_loader, epochs, task='jenis', model_name='model'):
    # Use weighted loss for imbalanced data
    if task == 'jenis':
        class_weights = jenis_weights
    else:  # warna
        class_weights = warna_weights
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
    
    # Learning Rate Scheduler - ReduceLROnPlateau (not aggressive)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',           # maximize validation accuracy
        factor=0.5,           # reduce LR by half (not aggressive)
        patience=5,           # wait 5 epochs before reducing
        verbose=True,
        min_lr=1e-6
    )
    
    # Early Stopping Config (not aggressive)
    best_val_acc = 0.0
    patience_counter = 0
    early_stop_patience = 10  # wait 10 epochs without improvement
    
    # Create directory for saving models
    os.makedirs('saved_models', exist_ok=True)
    best_model_path = f'saved_models/{model_name}_{task}_best.pth'
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for images, jenis, warna in tqdm(train_loader, leave=False, desc=f'Epoch {epoch+1}/{epochs}'):
            images = images.to(device)
            labels = jenis.to(device) if task == 'jenis' else warna.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        
        train_acc = 100 * correct / total
        avg_loss = running_loss / len(train_loader)
        
        # Evaluate on validation set every epoch
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, jenis, warna in val_loader:
                images = images.to(device)
                labels = jenis.to(device) if task == 'jenis' else warna.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        
        # Update learning rate scheduler
        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'train_acc': train_acc,
                'loss': avg_loss,
            }, best_model_path)
            best_indicator = ' ⭐ NEW BEST!'
        else:
            patience_counter += 1
            best_indicator = ''
        
        # Print progress every 5 epochs or when new best
        if (epoch + 1) % 5 == 0 or best_indicator:
            print(f'Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Train: {train_acc:.2f}% - Val: {val_acc:.2f}% - LR: {current_lr:.6f}{best_indicator}')
        
        # Early stopping check
        if patience_counter >= early_stop_patience:
            print(f'Early stopping triggered after {epoch+1} epochs. Best Val Acc: {best_val_acc:.2f}%')
            break
    
    # Load best model weights
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f'Loaded best model from epoch {checkpoint["epoch"]+1} with Val Acc: {checkpoint["val_acc"]:.2f}%')
    
    return model

def evaluate_model(model, loader, task='jenis'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, jenis, warna in loader:
            images = images.to(device)
            labels = jenis.to(device) if task == 'jenis' else warna.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

In [ ]:
def load_best_model(model, model_name, task):
    """Load best saved model weights"""
    checkpoint_path = f'saved_models/{model_name}_{task}_best.pth'
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f'Loaded {model_name} {task} - Epoch: {checkpoint["epoch"]+1}, Val Acc: {checkpoint["val_acc"]:.2f}%')
        return model
    else:
        print(f'No checkpoint found at {checkpoint_path}')
        return None

print('Training functions defined')

## 5. Train Jenis Models

In [ ]:
print('Training ResNet-50 for Jenis...')
resnet_jenis = ResNetModel(num_classes=2).to(device)
resnet_jenis = train_model(resnet_jenis, train_loader, epochs=30, task='jenis', model_name='resnet50')
acc = evaluate_model(resnet_jenis, val_loader, task='jenis')
print(f'Final ResNet-50 Jenis Accuracy: {acc:.2f}%\n')

print('Training EfficientNet-B0 for Jenis...')
efficientnet_jenis = EfficientNetModel(num_classes=2).to(device)
efficientnet_jenis = train_model(efficientnet_jenis, train_loader, epochs=30, task='jenis', model_name='efficientnet')
acc = evaluate_model(efficientnet_jenis, val_loader, task='jenis')
print(f'Final EfficientNet-B0 Jenis Accuracy: {acc:.2f}%\n')

print('Training MobileNet-V2 for Jenis...')
mobilenet_jenis = MobileNetModel(num_classes=2).to(device)
mobilenet_jenis = train_model(mobilenet_jenis, train_loader, epochs=30, task='jenis', model_name='mobilenet')
acc = evaluate_model(mobilenet_jenis, val_loader, task='jenis')
print(f'Final MobileNet-V2 Jenis Accuracy: {acc:.2f}%\n')

print('Training ResNet-18 for Jenis...')
resnet18_jenis = ResNet18Model(num_classes=2).to(device)
resnet18_jenis = train_model(resnet18_jenis, train_loader, epochs=30, task='jenis', model_name='resnet18')
acc = evaluate_model(resnet18_jenis, val_loader, task='jenis')
print(f'Final ResNet-18 Jenis Accuracy: {acc:.2f}%\n')

print('Training DenseNet-121 for Jenis...')
densenet_jenis = DenseNetModel(num_classes=2).to(device)
densenet_jenis = train_model(densenet_jenis, train_loader, epochs=30, task='jenis', model_name='densenet')
acc = evaluate_model(densenet_jenis, val_loader, task='jenis')
print(f'Final DenseNet-121 Jenis Accuracy: {acc:.2f}%\n')

print('Training ShuffleNet-V2 for Jenis...')
shufflenet_jenis = ShuffleNetModel(num_classes=2).to(device)
shufflenet_jenis = train_model(shufflenet_jenis, train_loader, epochs=30, task='jenis', model_name='shufflenet')
acc = evaluate_model(shufflenet_jenis, val_loader, task='jenis')
print(f'Final ShuffleNet-V2 Jenis Accuracy: {acc:.2f}%\n')

print('All Jenis models trained')

## 6. Train Warna Models

In [ ]:
print('Training ResNet-50 for Warna...')
resnet_warna = ResNetModel(num_classes=5).to(device)
resnet_warna = train_model(resnet_warna, train_loader, epochs=30, task='warna', model_name='resnet50')
acc = evaluate_model(resnet_warna, val_loader, task='warna')
print(f'Final ResNet-50 Warna Accuracy: {acc:.2f}%\n')

print('Training EfficientNet-B0 for Warna...')
efficientnet_warna = EfficientNetModel(num_classes=5).to(device)
efficientnet_warna = train_model(efficientnet_warna, train_loader, epochs=30, task='warna', model_name='efficientnet')
acc = evaluate_model(efficientnet_warna, val_loader, task='warna')
print(f'Final EfficientNet-B0 Warna Accuracy: {acc:.2f}%\n')

print('Training MobileNet-V2 for Warna...')
mobilenet_warna = MobileNetModel(num_classes=5).to(device)
mobilenet_warna = train_model(mobilenet_warna, train_loader, epochs=30, task='warna', model_name='mobilenet')
acc = evaluate_model(mobilenet_warna, val_loader, task='warna')
print(f'Final MobileNet-V2 Warna Accuracy: {acc:.2f}%\n')

print('Training ResNet-18 for Warna...')
resnet18_warna = ResNet18Model(num_classes=5).to(device)
resnet18_warna = train_model(resnet18_warna, train_loader, epochs=30, task='warna', model_name='resnet18')
acc = evaluate_model(resnet18_warna, val_loader, task='warna')
print(f'Final ResNet-18 Warna Accuracy: {acc:.2f}%\n')

print('Training DenseNet-121 for Warna...')
densenet_warna = DenseNetModel(num_classes=5).to(device)
densenet_warna = train_model(densenet_warna, train_loader, epochs=30, task='warna', model_name='densenet')
acc = evaluate_model(densenet_warna, val_loader, task='warna')
print(f'Final DenseNet-121 Warna Accuracy: {acc:.2f}%\n')

print('Training ShuffleNet-V2 for Warna...')
shufflenet_warna = ShuffleNetModel(num_classes=5).to(device)
shufflenet_warna = train_model(shufflenet_warna, train_loader, epochs=30, task='warna', model_name='shufflenet')
acc = evaluate_model(shufflenet_warna, val_loader, task='warna')
print(f'Final ShuffleNet-V2 Warna Accuracy: {acc:.2f}%\n')

print('All Warna models trained')

In [ ]:
print('\n' + '='*60)
print('SUMMARY - Individual Model Performance on Validation Set')
print('='*60)

print('\nJENIS Models:')
print(f"  ResNet-50:      {evaluate_model(resnet_jenis, val_loader, 'jenis'):.2f}%")
print(f"  ResNet-18:      {evaluate_model(resnet18_jenis, val_loader, 'jenis'):.2f}%")
print(f"  EfficientNet:   {evaluate_model(efficientnet_jenis, val_loader, 'jenis'):.2f}%")
print(f"  MobileNet:      {evaluate_model(mobilenet_jenis, val_loader, 'jenis'):.2f}%")
print(f"  DenseNet:       {evaluate_model(densenet_jenis, val_loader, 'jenis'):.2f}%")
print(f"  ShuffleNet:     {evaluate_model(shufflenet_jenis, val_loader, 'jenis'):.2f}%")

print('\nWARNA Models:')
print(f"  ResNet-50:      {evaluate_model(resnet_warna, val_loader, 'warna'):.2f}%")
print(f"  ResNet-18:      {evaluate_model(resnet18_warna, val_loader, 'warna'):.2f}%")
print(f"  EfficientNet:   {evaluate_model(efficientnet_warna, val_loader, 'warna'):.2f}%")
print(f"  MobileNet:      {evaluate_model(mobilenet_warna, val_loader, 'warna'):.2f}%")
print(f"  DenseNet:       {evaluate_model(densenet_warna, val_loader, 'warna'):.2f}%")
print(f"  ShuffleNet:     {evaluate_model(shufflenet_warna, val_loader, 'warna'):.2f}%")
print('='*60)

## 7. Feature Extraction & Meta-Learner Ensemble

### Meta-Learner (Stacking) Approach:

**Feature Extraction:**
- Extract probability distributions (softmax outputs) from all 6 pretrained models
- For Jenis: 6 models × 2 classes = 12 features per sample
- For Warna: 6 models × 5 classes = 30 features per sample
- These probability vectors capture the confidence of each base model

**Meta-Learners:**
1. **Logistic Regression**: Linear classifier on probability features
2. **SVM (RBF kernel)**: Non-linear decision boundary with support vectors
3. **KNN**: Distance-based voting in feature space

**Why Better than Majority Voting?**
- Learns optimal weights for each model's prediction
- Considers prediction confidence (probabilities), not just hard labels
- Can capture non-linear relationships between base models
- Trained specifically to maximize validation accuracy

In [ ]:
def extract_features(models, loader, task='jenis'):
    """Extract probability features from all models for stacking ensemble"""
    all_features = []
    all_labels = []
    
    for model in models:
        model.eval()
        model_probs = []
        
        with torch.no_grad():
            for images, jenis, warna in loader:
                images = images.to(device)
                outputs = model(images)
                # Get probability distributions (softmax)
                probs = torch.softmax(outputs, dim=1)
                model_probs.extend(probs.cpu().numpy())
                
                # Collect labels only once
                if len(all_labels) < len(loader.dataset):
                    labels = jenis if task == 'jenis' else warna
                    all_labels.extend(labels.numpy())
        
        all_features.append(model_probs)
    
    # Stack features: shape (n_samples, n_models * n_classes)
    stacked_features = np.hstack([np.array(f) for f in all_features])
    all_labels = np.array(all_labels)
    
    return stacked_features, all_labels

def train_meta_learner(features_train, labels_train, meta_learner_type='logistic'):
    """Train meta-learner on stacked features"""
    # Standardize features
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features_train)
    
    if meta_learner_type == 'logistic':
        meta_learner = LogisticRegression(
            max_iter=1000, 
            random_state=42,
            C=1.0,
            solver='lbfgs'
        )
    elif meta_learner_type == 'svm':
        meta_learner = SVC(
            kernel='rbf',
            C=1.0,
            gamma='scale',
            probability=True,
            random_state=42
        )
    elif meta_learner_type == 'knn':
        meta_learner = KNeighborsClassifier(
            n_neighbors=5,
            weights='distance',
            metric='minkowski'
        )
    else:
        raise ValueError(f"Unknown meta_learner_type: {meta_learner_type}")
    
    meta_learner.fit(features_scaled, labels_train)
    return meta_learner, scaler

def predict_with_meta_learner(models, loader, meta_learner, scaler, task='jenis', is_test=False):
    """Predict using meta-learner ensemble"""
    all_features = []
    
    for model in models:
        model.eval()
        model_probs = []
        
        with torch.no_grad():
            for batch in loader:
                # Handle different loader outputs (train/val has 3 values, test has 2)
                if is_test:
                    images, _ = batch  # test loader: (images, names)
                else:
                    images, jenis, warna = batch  # train/val loader: (images, jenis, warna)
                
                images = images.to(device)
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)
                model_probs.extend(probs.cpu().numpy())
        
        all_features.append(model_probs)
    
    # Stack and scale features
    stacked_features = np.hstack([np.array(f) for f in all_features])
    features_scaled = scaler.transform(stacked_features)
    
    # Predict with meta-learner
    predictions = meta_learner.predict(features_scaled)
    return predictions

print('Meta-learner ensemble functions defined')

## 7.1. Train Meta-Learners (Stacking Ensemble)

In [ ]:
jenis_models = [resnet_jenis, efficientnet_jenis, mobilenet_jenis, 
                resnet18_jenis, densenet_jenis, shufflenet_jenis]
warna_models = [resnet_warna, efficientnet_warna, mobilenet_warna,
                resnet18_warna, densenet_warna, shufflenet_warna]

print('Extracting features for JENIS models...')
# Use training set for meta-learner training
jenis_features_train, jenis_labels_train = extract_features(jenis_models, train_loader, task='jenis')
print(f'Jenis features shape: {jenis_features_train.shape}')

print('\nExtracting features for WARNA models...')
warna_features_train, warna_labels_train = extract_features(warna_models, train_loader, task='warna')
print(f'Warna features shape: {warna_features_train.shape}')

print('\n' + '='*60)
print('Training Meta-Learners')
print('='*60)

# Train different meta-learners for JENIS
print('\nJENIS Meta-Learners:')
print('  Training Logistic Regression...')
jenis_lr, jenis_lr_scaler = train_meta_learner(jenis_features_train, jenis_labels_train, 'logistic')
print('  Training SVM...')
jenis_svm, jenis_svm_scaler = train_meta_learner(jenis_features_train, jenis_labels_train, 'svm')
print('  Training KNN...')
jenis_knn, jenis_knn_scaler = train_meta_learner(jenis_features_train, jenis_labels_train, 'knn')

# Train different meta-learners for WARNA
print('\nWARNA Meta-Learners:')
print('  Training Logistic Regression...')
warna_lr, warna_lr_scaler = train_meta_learner(warna_features_train, warna_labels_train, 'logistic')
print('  Training SVM...')
warna_svm, warna_svm_scaler = train_meta_learner(warna_features_train, warna_labels_train, 'svm')
print('  Training KNN...')
warna_knn, warna_knn_scaler = train_meta_learner(warna_features_train, warna_labels_train, 'knn')

print('\nAll meta-learners trained successfully!')

## 7.2. Evaluate Meta-Learner Ensembles

In [ ]:
print('='*80)
print('VALIDATION RESULTS - Meta-Learner Ensembles')
print('='*80)

# Get true labels
all_labels_jenis = []
all_labels_warna = []
for _, jenis, warna in val_loader:
    all_labels_jenis.extend(jenis.numpy())
    all_labels_warna.extend(warna.numpy())

# Evaluate Logistic Regression
print('\n1. Logistic Regression Ensemble:')
jenis_pred_lr = predict_with_meta_learner(jenis_models, val_loader, jenis_lr, jenis_lr_scaler, 'jenis')
warna_pred_lr = predict_with_meta_learner(warna_models, val_loader, warna_lr, warna_lr_scaler, 'warna')
jenis_acc_lr = accuracy_score(all_labels_jenis, jenis_pred_lr)
warna_acc_lr = accuracy_score(all_labels_warna, warna_pred_lr)
emr_lr = sum([1 for i in range(len(jenis_pred_lr)) 
              if jenis_pred_lr[i] == all_labels_jenis[i] and warna_pred_lr[i] == all_labels_warna[i]]) / len(jenis_pred_lr)
print(f'   Jenis Accuracy: {jenis_acc_lr:.4f} ({jenis_acc_lr*100:.2f}%)')
print(f'   Warna Accuracy: {warna_acc_lr:.4f} ({warna_acc_lr*100:.2f}%)')
print(f'   Exact Match Ratio (EMR): {emr_lr:.4f} ({emr_lr*100:.2f}%)')

# Evaluate SVM
print('\n2. SVM Ensemble:')
jenis_pred_svm = predict_with_meta_learner(jenis_models, val_loader, jenis_svm, jenis_svm_scaler, 'jenis')
warna_pred_svm = predict_with_meta_learner(warna_models, val_loader, warna_svm, warna_svm_scaler, 'warna')
jenis_acc_svm = accuracy_score(all_labels_jenis, jenis_pred_svm)
warna_acc_svm = accuracy_score(all_labels_warna, warna_pred_svm)
emr_svm = sum([1 for i in range(len(jenis_pred_svm)) 
               if jenis_pred_svm[i] == all_labels_jenis[i] and warna_pred_svm[i] == all_labels_warna[i]]) / len(jenis_pred_svm)
print(f'   Jenis Accuracy: {jenis_acc_svm:.4f} ({jenis_acc_svm*100:.2f}%)')
print(f'   Warna Accuracy: {warna_acc_svm:.4f} ({warna_acc_svm*100:.2f}%)')
print(f'   Exact Match Ratio (EMR): {emr_svm:.4f} ({emr_svm*100:.2f}%)')

# Evaluate KNN
print('\n3. KNN Ensemble:')
jenis_pred_knn = predict_with_meta_learner(jenis_models, val_loader, jenis_knn, jenis_knn_scaler, 'jenis')
warna_pred_knn = predict_with_meta_learner(warna_models, val_loader, warna_knn, warna_knn_scaler, 'warna')
jenis_acc_knn = accuracy_score(all_labels_jenis, jenis_pred_knn)
warna_acc_knn = accuracy_score(all_labels_warna, warna_pred_knn)
emr_knn = sum([1 for i in range(len(jenis_pred_knn)) 
               if jenis_pred_knn[i] == all_labels_jenis[i] and warna_pred_knn[i] == all_labels_warna[i]]) / len(jenis_pred_knn)
print(f'   Jenis Accuracy: {jenis_acc_knn:.4f} ({jenis_acc_knn*100:.2f}%)')
print(f'   Warna Accuracy: {warna_acc_knn:.4f} ({warna_acc_knn*100:.2f}%)')
print(f'   Exact Match Ratio (EMR): {emr_knn:.4f} ({emr_knn*100:.2f}%)')

# Find best ensemble
best_emr = max(emr_lr, emr_svm, emr_knn)
best_method = ['Logistic Regression', 'SVM', 'KNN'][[emr_lr, emr_svm, emr_knn].index(best_emr)]

print('\n' + '='*80)
print(f'BEST ENSEMBLE: {best_method} with EMR: {best_emr:.4f} ({best_emr*100:.2f}%)')
print('='*80)

## 7.3. Comparison: Meta-Learner vs Majority Voting

In [ ]:
# Majority Voting Ensemble (baseline)
def majority_voting_predict(models, loader, task, is_test=False):
    predictions_list = []
    for model in models:
        model.eval()
        preds = []
        with torch.no_grad():
            for batch in loader:
                # Handle different loader outputs
                if is_test:
                    images, _ = batch  # test loader: (images, names)
                else:
                    images, _, _ = batch  # train/val loader: (images, jenis, warna)
                
                images = images.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                preds.extend(predicted.cpu().numpy())
        predictions_list.append(preds)
    
    ensemble_preds = []
    for i in range(len(predictions_list[0])):
        votes = [predictions_list[j][i] for j in range(len(models))]
        ensemble_preds.append(max(set(votes), key=votes.count))
    return np.array(ensemble_preds)

print('Evaluating Majority Voting (Baseline)...')
jenis_pred_voting = majority_voting_predict(jenis_models, val_loader, 'jenis')
warna_pred_voting = majority_voting_predict(warna_models, val_loader, 'warna')
jenis_acc_voting = accuracy_score(all_labels_jenis, jenis_pred_voting)
warna_acc_voting = accuracy_score(all_labels_warna, warna_pred_voting)
emr_voting = sum([1 for i in range(len(jenis_pred_voting)) 
                  if jenis_pred_voting[i] == all_labels_jenis[i] and warna_pred_voting[i] == all_labels_warna[i]]) / len(jenis_pred_voting)

print('\n' + '='*80)
print('ENSEMBLE COMPARISON')
print('='*80)
print(f'\n{"Method":<25} {"Jenis Acc":<15} {"Warna Acc":<15} {"EMR":<15}')
print('-'*80)
print(f'{"Majority Voting":<25} {jenis_acc_voting*100:>6.2f}%        {warna_acc_voting*100:>6.2f}%        {emr_voting*100:>6.2f}%')
print(f'{"Logistic Regression":<25} {jenis_acc_lr*100:>6.2f}%        {warna_acc_lr*100:>6.2f}%        {emr_lr*100:>6.2f}%')
print(f'{"SVM":<25} {jenis_acc_svm*100:>6.2f}%        {warna_acc_svm*100:>6.2f}%        {emr_svm*100:>6.2f}%')
print(f'{"KNN":<25} {jenis_acc_knn*100:>6.2f}%        {warna_acc_knn*100:>6.2f}%        {emr_knn*100:>6.2f}%')
print('='*80)

# Calculate improvements
improvements = {
    'Logistic Regression': (emr_lr - emr_voting) * 100,
    'SVM': (emr_svm - emr_voting) * 100,
    'KNN': (emr_knn - emr_voting) * 100
}

print('\nImprovement over Majority Voting:')
for method, improvement in improvements.items():
    sign = '+' if improvement > 0 else ''
    print(f'  {method:<25} {sign}{improvement:.2f}%')

best_meta = max(improvements, key=improvements.get)
print(f'\nBest Meta-Learner: {best_meta}')

In [ ]:
jenis_models = [resnet_jenis, efficientnet_jenis, mobilenet_jenis, 
                resnet18_jenis, densenet_jenis, shufflenet_jenis]
warna_models = [resnet_warna, efficientnet_warna, mobilenet_warna,
                resnet18_warna, densenet_warna, shufflenet_warna]

jenis_acc, warna_acc, emr = evaluate_ensemble(jenis_models, warna_models, val_loader)

print(f'Ensemble Validation Results (6 models):')
print(f'Jenis Accuracy: {jenis_acc:.4f}')
print(f'Warna Accuracy: {warna_acc:.4f}')
print(f'Exact Match Ratio (EMR): {emr:.4f}')

## 8. Test Prediction

In [ ]:
test_dir = './test/test'
test_image_files = [f for f in os.listdir(test_dir) if f.endswith(('.jpg', '.png'))]

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, image_files, transform=None):
        self.image_files = image_files
        self.transform = transform
        self.clahe_transform = CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8))
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(test_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        # Apply CLAHE
        image = self.clahe_transform(image)
        
        if self.transform:
            image = self.transform(image)
        
        return image, img_name

test_dataset = TestDataset(test_image_files, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

In [ ]:
# Get image names
image_names = []
for _, names in test_loader:
    image_names.extend(names)

print('Generating predictions with best meta-learners...')
print('Using Logistic Regression for final predictions')

# Predict with best meta-learner (Logistic Regression)
jenis_preds = predict_with_meta_learner(jenis_models, test_loader, jenis_lr, jenis_lr_scaler, 'jenis', is_test=True)
warna_preds = predict_with_meta_learner(warna_models, test_loader, warna_lr, warna_lr_scaler, 'warna', is_test=True)

# Create submission for meta-learner ensemble
submission_meta = pd.DataFrame({
    'id': [name.split('.')[0] for name in image_names],
    'jenis': jenis_preds,
    'warna': warna_preds
})

submission_meta.to_csv('submission_metalearner_ensemble.csv', index=False)
print('\nSubmission saved to submission_metalearner_ensemble.csv')
print(submission_meta.head(10))

# Also create baseline majority voting submission
print('\n\nGenerating baseline majority voting predictions...')
jenis_preds_voting = majority_voting_predict(jenis_models, test_loader, 'jenis', is_test=True)
warna_preds_voting = majority_voting_predict(warna_models, test_loader, 'warna', is_test=True)

submission_voting = pd.DataFrame({
    'id': [name.split('.')[0] for name in image_names],
    'jenis': jenis_preds_voting,
    'warna': warna_preds_voting
})

submission_voting.to_csv('submission_majority_voting.csv', index=False)
print('\nBaseline submission saved to submission_majority_voting.csv')
print(submission_voting.head(10))